# 🚀 SAARA — Complete Notebook

> **SAARA v1.7.0** · Document → AI Training Dataset Factory

**What SAARA does:**
```
Documents / Prompts
      │
      ├─▶  Extract Text  (PDF, TXT, MD, JSONL)
      ├─▶  Label Data    (LLM generates Q&A / instruction pairs)
      ├─▶  Synthesize    (Teacher model → distillation dataset)
      ├─▶  Filter        (Quality gate, dedup)
      └─▶  Format        (Alpaca · ChatML · ShareGPT · DPO · Completion)
```

> ⚠️ **Scope**: SAARA is a *data* tool. Training is deliberately out of scope — use [Axolotl](https://github.com/OpenAccess-AI-Collective/axolotl), [LLaMA-Factory](https://github.com/hiyouga/LLaMA-Factory), or [Unsloth](https://github.com/unslothai/unsloth) with SAARA's output.

---

## 📦  Installation

In [ ]:
!pip install saara-ai pdfplumber pymupdf

# Ollama (CPU-friendly, recommended)
# → Install: https://ollama.com/download
# → Terminal: ollama serve
# → Terminal: ollama pull mistral

# vLLM (GPU, high-throughput)
# → !pip install vllm    (needs CUDA GPU)

In [ ]:
import saara
print("SAARA", saara.__version__)
print("Public API:", saara.__all__)

---
## ⚙️  1 · Backend Setup

In [ ]:
from saara import quickapi

# ── Ollama ── CPU-friendly, local, no GPU required ─────────────────────────
# Requires: ollama serve  +  ollama pull <model>
quickapi.setup(
    "ollama",
    model="mistral",          # or: llama3, granite3.1-dense:8b, qwen2.5, phi3
    temperature=0.7,
    max_tokens=2048,
    output_dir="./saara_output",
    verbose=True
)

In [ ]:
# ── vLLM ── GPU server, 5-10× faster throughput ────────────────────────────
# Uncomment to switch:

# quickapi.setup(
#     "vllm",
#     model="mistralai/Mistral-7B-Instruct-v0.2",  # any HuggingFace model ID
#     temperature=0.7,
#     max_tokens=2048,
#     output_dir="./saara_output",
#     verbose=True
# )

In [ ]:
# ── Auto-detect ── tries Ollama, falls back to vLLM ───────────────────────
# quickapi.setup("auto", model="mistral")

# Check what's available
quickapi.check_backends()

# Current status
print(quickapi.get_status())

---
## 📄  2 · Extract Text from Documents

In [ ]:
# ── PDF extraction ─────────────────────────────────────────────────────────
pdf_data = quickapi.dataExtract_PDF(
    "sample.pdf",
    use_ocr=True,       # OCR for scanned pages
    save_output=True
)
print(f"Pages  : {pdf_data.get('total_pages', 0)}")
print(f"Chunks : {len(pdf_data.get('chunks', []))}")
print(f"Tables : {len(pdf_data.get('tables', []))}")
# Preview first chunk
chunks = pdf_data.get('chunks', [])
if chunks:
    print("\nFirst chunk:")
    print(chunks[0]['text'][:300])

In [ ]:
# ── Text / Markdown / JSONL extraction ─────────────────────────────────────
# Create demo text file
sample_text = """Python is a high-level, general-purpose programming language.
It emphasizes code readability and supports multiple paradigms: procedural,
object-oriented, and functional. Created by Guido van Rossum in 1991,
Python has become one of the world's most popular languages, powering
web development, data science, AI/ML, automation, and scientific computing.

Key strengths: readable syntax, vast standard library, dynamic typing,
garbage collection, strong community, and cross-platform compatibility.
Notable frameworks include Django, FastAPI, NumPy, PyTorch, and TensorFlow."""

with open("sample_doc.txt", "w") as f:
    f.write(sample_text)

text_data = quickapi.dataExtract_Text("sample_doc.txt", save_output=False)
print(f"Chunks     : {len(text_data.get('chunks', []))}")
print(f"Word count : {text_data.get('word_count', 0)}")

# Short alias
text_data = quickapi.extract("sample_doc.txt")

---
## 🏷️  3 · AI Data Labeling

In [ ]:
# LLM reads each chunk and generates instruction-response pairs
labeled = quickapi.dataLabel_Dataset(
    text_data,
    label_types=["qa", "instruction"],  # qa = Q&A, instruction = task pairs
    save_output=True
)
print(f"Labeled items : {labeled.get('total_labeled', 0)}")
print()
for item in labeled.get('labeled_items', [])[:3]:
    print(f"  Instruction : {item.get('instruction', '')[:90]}")
    print(f"  Response    : {item.get('response', '')[:90]}")
    print()

In [ ]:
# DataLabeler class — granular per-document operations
from saara import DataLabeler

labeler = DataLabeler({"model": "mistral", "base_url": "http://localhost:11434"})
# For vLLM: {"model": "mistralai/Mistral-7B-Instruct-v0.2", "base_url": "http://localhost:8000"}

text = """The attention mechanism in transformers assigns weighted importance to
every other token when encoding each token. Self-attention allows the model to
relate different positions of a sequence to capture contextual dependencies."""

print("=== Topic Extraction ===")
print(labeler.extract_topics(text))

print("\n=== Document Classification ===")
print(labeler.classify_document(text))

print("\n=== Quality Assessment ===")
print(labeler.assess_quality(text))

print("\n=== Summary ===")
result = labeler.summarize(text)
print(result.get('summary', ''))

print("\n=== Named Entities ===")
for e in labeler.extract_entities(text).get('entities', []):
    print(f"  {e['text']:25s}  [{e['type']}]")

---
## 🧬  4 · Synthetic Data Generation (4 Types)

In [ ]:
from saara import SyntheticDataGenerator, DataType

# Ollama config
synth_cfg = {"model": "mistral", "base_url": "http://localhost:11434"}
# vLLM config (uncomment):
# synth_cfg = {"model": "mistralai/Mistral-7B-Instruct-v0.2", "base_url": "http://localhost:8000"}

synth = SyntheticDataGenerator(synth_cfg)

source = """BERT (Bidirectional Encoder Representations from Transformers) is pre-trained
on Masked Language Modeling and Next Sentence Prediction tasks. It introduced deeply
bidirectional representations, outperforming left-to-right models. Fine-tuned BERT
set state-of-the-art results across 11 NLP benchmarks when published in 2018."""

for dtype in [DataType.FACTUAL, DataType.REASONING,
              DataType.CONVERSATIONAL, DataType.INSTRUCTION]:
    samples = synth.generate(
        text=source,
        data_types=[dtype],
        pairs_per_type=2,
        min_quality=0.6
    )
    print(f"\n{'─'*55}")
    print(f"  {dtype.value.upper()} ({len(samples)} pairs)")
    print(f"{'─'*55}")
    for s in samples:
        print(f"  I: {s.instruction[:100]}")
        print(f"  O: {s.output[:100]}")
        print()

---
## 🔬  5 · Model Distillation — Teacher → Dataset

In [ ]:
# ── 5A: List of prompts (Ollama) ───────────────────────────────────────────
result = quickapi.dataGenerate_Distillation(
    seed_prompts=[
        "Explain the difference between supervised and unsupervised learning.",
        "What is gradient descent and how does it work?",
        "Describe the attention mechanism in transformer models.",
        "What are the advantages of batch normalization?",
        "Explain the vanishing gradient problem and solutions.",
    ],
    format="alpaca",
    responses_per_prompt=1,
    save_output=True
)
print(f"Prompts  : {result['total_prompts']}")
print(f"Samples  : {result['total_samples']}")
print(f"Output   : {result['output_file']}")

print("\n── Sample ──")
item = result['items'][0]
print("Instruction:", item.get('instruction', '')[:120])
print("Output     :", item.get('output', '')[:200])

In [ ]:
# ── 5B: vLLM — faster for large batches ───────────────────────────────────
# Switch to vLLM before running this cell:

# quickapi.setup("vllm", model="mistralai/Mistral-7B-Instruct-v0.2")
#
# result = quickapi.dataGenerate_Distillation(
#     seed_prompts=[f"Explain ML concept #{i}" for i in range(500)],
#     format="alpaca",
#     responses_per_prompt=1,
#     save_output=True
# )
# print(f"Generated {result['total_samples']} samples at vLLM speed")

In [ ]:
# ── 5C: Diversity mode — multiple responses per prompt ─────────────────────
div = quickapi.dataGenerate_Distillation(
    seed_prompts=[
        "Explain overfitting and how to prevent it.",
        "What is transfer learning and when to use it?",
    ],
    format="chatml",
    responses_per_prompt=3,      # 3 slightly different responses
    diversity_mode=True,         # varies temperature for diversity
    system_prompt="You are a precise ML educator.",
    save_output=True
)
print(f"2 prompts × 3 responses = {div['total_samples']} samples")

In [ ]:
# ── 5D: DPO dataset — chosen / rejected pairs ─────────────────────────────
dpo = quickapi.dataGenerate_Distillation(
    seed_prompts=[
        "Compare SQL vs NoSQL databases.",
        "Explain the CAP theorem in distributed systems.",
        "What are microservices and their tradeoffs?",
    ],
    format="dpo",
    responses_per_prompt=2,      # longer = chosen, shorter = rejected
    diversity_mode=True,
    save_output=True
)
print(f"DPO pairs: {dpo['total_samples']}")
print("Keys per item:", list(dpo['items'][0].keys()))

In [ ]:
# ── 5E: Load prompts from a .txt file ────────────────────────────────────
with open("prompts.txt", "w") as f:
    f.write("What is the difference between a list and a tuple in Python?\n")
    f.write("Explain Python's GIL (Global Interpreter Lock).\n")
    f.write("What are Python decorators and when should you use them?\n")
    f.write("How does Python's garbage collection work?\n")

file_result = quickapi.dataGenerate_Distillation(
    seed_prompts="prompts.txt",  # one prompt per line
    format="alpaca",
    save_output=True
)
print(f"From file: {file_result['total_samples']} samples")

In [ ]:
# ── 5F: Load prompts from a .jsonl file ──────────────────────────────────
import json
with open("prompts.jsonl", "w") as f:
    for p in ["Explain REST API design principles.",
               "What is Docker containerization?",
               "Describe CI/CD pipelines."]:
        f.write(json.dumps({"instruction": p}) + "\n")

jsonl_result = quickapi.dataGenerate_Distillation(
    seed_prompts="prompts.jsonl",
    format="sharegpt",
    system_prompt="You are a senior software engineer.",
    save_output=True
)
print(f"From JSONL: {jsonl_result['total_samples']} samples")

In [ ]:
# ── 5G: Short alias — synthesize() ────────────────────────────────────────
r = quickapi.synthesize(
    ["Explain the SOLID principles.", "What is dependency injection?"],
    format="alpaca"
)
print(f"synthesize(): {r['total_samples']} samples")

---
## 🎚️  6 · Quality Filtering

In [ ]:
# Remove low-quality, duplicate, or too-short samples
filtered = quickapi.dataDistill_Dataset(
    labeled,           # any labeled/distillation result dict  OR  a .jsonl path
    save_output=True
)
print(f"Before : {labeled.get('total_labeled', 0)} items")
print(f"After  : {filtered.get('total_distilled', 0)} items")
print(f"Removed: {filtered.get('removed_count', 0)} items")

# Short alias
filtered = quickapi.distill(labeled)

In [ ]:
# LLM-based quality judge — 5-dimension scoring
from saara import QualityJudge

judge = QualityJudge({"model": "mistral", "base_url": "http://localhost:11434"})

score = judge.judge(
    instruction="What is backpropagation?",
    response=(
        "Backpropagation computes gradients by applying the chain rule backwards "
        "through each layer, from output to input, to update network weights."
    )
)

print("Quality scores:")
for k, v in score.items():
    print(f"  {k:20s}: {v}")

---
## 🔄  7 · Format Conversion

In [ ]:
# Reference data
raw = {
    "labeled_items": [
        {"instruction": "What is a neural network?",
         "response": "A neural network is a computational model inspired by the brain, "
                     "composed of layers of interconnected nodes that learn representations."},
        {"instruction": "Explain the ReLU activation function.",
         "response": "ReLU computes max(0, x), introducing non-linearity while avoiding "
                     "the vanishing gradient problem that sigmoid/tanh suffer from."},
        {"instruction": "What is dropout regularization?",
         "response": "Dropout randomly deactivates neurons during training, preventing "
                     "co-adaptation and reducing overfitting as an ensemble method."},
    ],
    "total_labeled": 3
}

print("Supported formats:", quickapi.list_formats())

In [ ]:
# ── Alpaca ──
alpaca = quickapi.dataConvert_Format(raw, target_format="alpaca")
print("ALPACA → keys:", list(alpaca['items'][0].keys()))
print(alpaca['items'][0])

In [ ]:
# ── ChatML ──
chatml = quickapi.dataConvert_Format(
    raw, target_format="chatml",
    system_prompt="You are a machine learning tutor."
)
print("CHATML → keys:", list(chatml['items'][0].keys()))
print(chatml['items'][0])

In [ ]:
# ── ShareGPT ──
sharegpt = quickapi.dataConvert_Format(
    raw, target_format="sharegpt",
    system_prompt="You are a helpful AI assistant."
)
print("SHAREGPT → keys:", list(sharegpt['items'][0].keys()))
print(sharegpt['items'][0])

In [ ]:
# ── Completion ──
completion = quickapi.dataConvert_Format(raw, target_format="completion")
print("COMPLETION → keys:", list(completion['items'][0].keys()))
print(completion['items'][0])

In [ ]:
# ── DPO ── (needs chosen + rejected)
dpo_raw = {
    "labeled_items": [
        {"instruction": "Explain softmax.",
         "response": "Softmax converts raw logits into a probability distribution "
                     "that sums to 1, making outputs interpretable as class probabilities.",
         "rejected": "Softmax is a function used in neural networks."},
        {"instruction": "What is layer normalization?",
         "response": "Layer normalization normalizes activations across features within "
                     "a single sample, stabilizing training in transformers.",
         "rejected": "It normalizes layer outputs."},
    ],
    "total_labeled": 2
}
dpo = quickapi.dataConvert_Format(dpo_raw, target_format="dpo")
print("DPO → keys:", list(dpo['items'][0].keys()))
print(dpo['items'][0])

In [ ]:
# Convert an existing JSONL file to a different format (no LLM needed)
# result = quickapi.convert_existing("existing_data.jsonl", target_format="sharegpt")

---
## 🚀  8 · One-Call Pipelines

In [ ]:
# PDF → Alpaca in one line
r = quickapi.pdf_to_dataset("sample.pdf", format="alpaca", min_quality=0.7)
print(f"PDF pipeline → {r['total_items']} items → {r['output_file']}")

In [ ]:
# Text → ChatML in one line
r = quickapi.text_to_dataset(
    "sample_doc.txt",
    format="chatml",
    system_prompt="You are a Python expert."
)
print(f"Text pipeline → {r['total_items']} items → {r['output_file']}")

---
## 🤖  9 · RAG Engine

In [ ]:
from saara import create_rag_engine

rag = create_rag_engine(
    documents=["sample_doc.txt"],
    config={
        "model": "mistral",
        "base_url": "http://localhost:11434",
        # vLLM: "base_url": "http://localhost:8000"
        "chunk_size": 512,
        "top_k": 3
    }
)

for question in [
    "Who created Python?",
    "What programming paradigms does Python support?",
    "Name some Python frameworks.",
]:
    answer = rag.query(question)
    print(f"Q: {question}")
    print(f"A: {answer.get('answer', '')[:200]}")
    print()

In [ ]:
# Generate a labeled dataset from your RAG engine
rag_data = rag.generate_dataset(num_questions=10, format="alpaca")
print(f"RAG dataset: {len(rag_data)} items")

---
## 🔧  10 · DataPipeline Class

In [ ]:
from saara import DataPipeline, PipelineConfig

cfg = PipelineConfig(
    model="mistral",
    backend="ollama",
    output_dir="./pipeline_output",
    output_format="alpaca",
    chunk_size=512,
    min_quality=0.7,
    temperature=0.7
)

pipeline = DataPipeline(cfg.to_dict())
result = pipeline.run("sample_doc.txt")
print(f"Pipeline: {result.get('total_samples', 0)} samples → {result.get('output_file', '')}")

In [ ]:
# YAML config — great for reproducible runs
yaml_cfg = """model: mistral
backend: ollama
output_dir: ./yaml_output
output_format: sharegpt
chunk_size: 400
min_quality: 0.75
temperature: 0.6
system_prompt: "You are a helpful AI assistant."
"""
with open("pipeline.yaml", "w") as f:
    f.write(yaml_cfg)

p2 = DataPipeline("pipeline.yaml")
r2 = p2.run("sample_doc.txt")
print(f"YAML pipeline: {r2.get('total_samples', 0)} samples")

---
## 🛠️  11 · Utilities

In [ ]:
from saara import load_jsonl, save_jsonl, split_dataset

data = [
    {"instruction": f"Question {i}", "output": f"Answer {i}"}
    for i in range(1, 11)
]

# Save
save_jsonl(data, "dataset.jsonl")
print(f"Saved {len(data)} items")

# Load
loaded = load_jsonl("dataset.jsonl")
print(f"Loaded {len(loaded)} items")

# Split
splits = split_dataset(loaded, train=0.8, val=0.1, test=0.1)
print(f"Train {len(splits['train'])} | Val {len(splits['val'])} | Test {len(splits['test'])}")

In [ ]:
# Backend & status utilities
quickapi.check_backends()
print(quickapi.get_status())
print(quickapi.list_formats())

---
## ✅  12 · Complete End-to-End Workflow

In [ ]:
from saara import quickapi, load_jsonl, save_jsonl, split_dataset

print("══ SAARA · Full Pipeline ══\n")

# 1 · Setup
print("[1/5] Backend setup")
quickapi.setup("ollama", model="mistral", verbose=False)
# For vLLM: quickapi.setup("vllm", model="mistralai/Mistral-7B-Instruct-v0.2")

# 2 · Extract
print("[2/5] Extracting text")
doc = quickapi.extract("sample_doc.txt")
print(f"      {len(doc.get('chunks', []))} chunks")

# 3 · Label
print("[3/5] Labeling (Q&A + instruction pairs)")
labeled = quickapi.label(doc, label_types=["qa", "instruction"])
print(f"      {labeled.get('total_labeled', 0)} pairs generated")

# 4 · Synthesize (teacher model distillation)
print("[4/5] Synthesizing distillation data")
synth = quickapi.synthesize(
    ["What is Python?", "List Python's key strengths.",
     "What are Python's main use cases?"],
    format="alpaca",
    save_output=False
)
print(f"      {synth['total_samples']} distillation samples")

# 5 · Filter + Format + Split
print("[5/5] Filter → Format → Split")
clean   = quickapi.distill(labeled)
alpaca  = quickapi.convert(clean, format="alpaca")

all_items = alpaca.get('items', []) + synth.get('items', [])
splits    = split_dataset(all_items, train=0.8, val=0.1, test=0.1)

save_jsonl(splits['train'], "./saara_output/train.jsonl")
save_jsonl(splits['val'],   "./saara_output/val.jsonl")
save_jsonl(splits['test'],  "./saara_output/test.jsonl")

print(f"\n══ Done ══")
print(f"  Total   : {len(all_items)} samples")
print(f"  Train   : {len(splits['train'])} → train.jsonl")
print(f"  Val     : {len(splits['val'])}   → val.jsonl")
print(f"  Test    : {len(splits['test'])}  → test.jsonl")
print()
print("  Plug into Axolotl / LLaMA-Factory / Unsloth for fine-tuning.")

---
## 📋  Quick Reference

### Functions

| Function | What it does |
|---|---|
| `quickapi.setup(backend, model)` | Initialize backend once |
| `quickapi.extract(file)` | Extract text from PDF / TXT / MD / JSONL |
| `quickapi.label(data)` | LLM-generated Q&A + instruction pairs |
| `quickapi.synthesize(prompts)` | Teacher model → distillation dataset |
| `quickapi.dataGenerate_Distillation(prompts)` | Full distillation API |
| `quickapi.distill(data)` | Quality filter + dedup |
| `quickapi.convert(data, format)` | Convert to any output format |
| `quickapi.pdf_to_dataset(file)` | Full PDF → dataset in 1 call |
| `quickapi.text_to_dataset(file)` | Full text → dataset in 1 call |
| `quickapi.check_backends()` | Check Ollama / vLLM availability |
| `quickapi.list_formats()` | Show supported formats |
| `quickapi.get_status()` | Current config |
| `create_rag_engine(docs, config)` | Document Q&A system |
| `DataPipeline(config)` | Full pipeline class |
| `DataLabeler(config)` | Per-document labeling |
| `SyntheticDataGenerator(config)` | Typed synthetic data |
| `load_jsonl(path)` | Load JSONL |
| `save_jsonl(data, path)` | Save to JSONL |
| `split_dataset(data, train, val, test)` | Train/val/test split |

### Backends

| Backend | Setup | Best For |
|---|---|---|
| `"ollama"` | `ollama serve` + `ollama pull <model>` | CPU, local dev |
| `"vllm"` | `pip install vllm` + CUDA GPU | GPU servers, large batches |
| `"auto"` | Tries Ollama → falls back to vLLM | Portable scripts |

### Output Formats

| Format | Use Case |
|---|---|
| `alpaca` | General instruction fine-tuning |
| `chatml` | OpenAI-style chat fine-tuning |
| `sharegpt` | Multi-turn conversation fine-tuning |
| `dpo` | Direct Preference Optimization (chosen/rejected) |
| `completion` | Text completion / continuation tasks |